## Feature Importance — Built-in (XGBoost)

In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

xgb_balanced = joblib.load("xgb_balanced.joblib")

df = df = pd.read_csv("cleaned_data.csv", index_col="id")

x = df.drop(columns=["default_payment_next_month"])
y = df["default_payment_next_month"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [2]:
import pandas as pd

xgb_final_model = xgb_balanced

feature_names = xgb_final_model.named_steps["preprocessor"].get_feature_names_out()

feature_importance = xgb_final_model.named_steps["classifier"].feature_importances_

fi = pd.Series(feature_importance, feature_names).sort_values(ascending=False)

fi

remainder__pay_1        0.288600
remainder__pay_2        0.248418
remainder__pay_3        0.060246
remainder__pay_4        0.048776
remainder__pay_5        0.041603
remainder__pay_6        0.033244
num__pay_amt2           0.029001
num__limit_bal          0.024598
num__pay_amt1           0.023576
num__pay_amt3           0.023177
num__bill_amt1          0.021580
num__pay_amt4           0.017046
num__pay_amt5           0.015600
num__pay_amt6           0.014853
num__bill_amt2          0.013909
remainder__education    0.013904
num__bill_amt3          0.013112
remainder__marriage     0.012807
num__bill_amt5          0.012478
num__bill_amt4          0.012344
num__bill_amt6          0.010538
num__age                0.010450
remainder__sex          0.010138
dtype: float32

### What this importance actually measures?
- Feature importance in XGBoost quantifies how much each input feature contributes to the model's prediction. helping interpret model and guide feature selection.

### Why this can be misleading?
- BEcause it depends on model if model overfits then its feature importance can not be trusted, Also feature importance is biased toward high cardinality or continuous feature which provide more opportunities to split the data but may not be able to create meaningful split.

### Why correlated features distort this view?
- when features are correlated models can use them interchangeably which can cause importance to be spread across group of correlated features.
- Removing one correlated feature may not degrade performance because other can "pick-up" the signal this affect importance of uncorrelated feature as they might be in top rankings of importance due to some feature which is totally redundant and can be removed. 

## Permutation Importance — Model-Agnostic

In [3]:
from sklearn.inspection import permutation_importance

r = permutation_importance(
    xgb_final_model,
    x_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='roc_auc'
)
prem_importance = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)

prem_importance

num__bill_amt4          0.070998
num__limit_bal          0.023360
num__pay_amt4           0.013110
num__bill_amt5          0.005926
num__pay_amt5           0.004712
num__pay_amt3           0.003726
num__bill_amt1          0.003527
num__pay_amt1           0.003317
num__pay_amt2           0.003277
num__bill_amt6          0.002795
remainder__pay_3        0.002478
remainder__pay_2        0.002478
remainder__pay_1        0.002211
remainder__sex          0.001967
num__pay_amt6           0.001898
num__bill_amt2          0.001643
remainder__marriage     0.001281
remainder__pay_4        0.000864
remainder__pay_6        0.000572
num__age                0.000481
num__bill_amt3         -0.000120
remainder__education   -0.000183
remainder__pay_5       -0.000257
dtype: float64

In [4]:
(prem_importance - fi).sort_values(ascending=True)

remainder__pay_1       -0.286389
remainder__pay_2       -0.245941
remainder__pay_3       -0.057769
remainder__pay_4       -0.047913
remainder__pay_5       -0.041860
remainder__pay_6       -0.032673
num__pay_amt2          -0.025724
num__pay_amt1          -0.020258
num__pay_amt3          -0.019451
num__bill_amt1         -0.018053
remainder__education   -0.014087
num__bill_amt3         -0.013233
num__pay_amt6          -0.012955
num__bill_amt2         -0.012266
remainder__marriage    -0.011526
num__pay_amt5          -0.010888
num__age               -0.009969
remainder__sex         -0.008171
num__bill_amt6         -0.007743
num__bill_amt5         -0.006551
num__pay_amt4          -0.003936
num__limit_bal         -0.001238
num__bill_amt4          0.058654
dtype: float64

- Permutation importance is calculated by shuffling the values of a single feature in the evaluation data (while keeping the trained model fixed) and measuring how much the model’s performance degrades. This process is repeated multiple times and averaged. Features whose shuffling causes a larger drop in performance are considered more important.

### Which features stay important?
- `bill_amt4`, `pay_amt4` and `limit_bal` stay important while others losses their importance.

### Which drop?
- `pay_1`, `pay_2` and `pay_3` looses their importance.

### What does this say about redundancy?
- If permuting a feature does not significantly hurts the performance, it suggests the feature's importance is largely duplicated by others and feature is redundant rather than uniquely important.